---
## 0 — Installation & Imports

In [ ]:
!pip install scikit-learn matplotlib seaborn scipy xgboost tqdm optuna shap --quiet
!pip install lightgbm --config-settings=cmake.define.USE_GPU=ON --quiet 2>/dev/null || \
    pip install lightgbm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings, subprocess, os
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, brier_score_loss,
    log_loss, classification_report, RocCurveDisplay,
    PrecisionRecallDisplay, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import lightgbm as lgb

np.random.seed(42)

---
## 1 — Configuration

In [ ]:
DATA_PATH    = "/content/drive/MyDrive/collab/f2/india+portugal+historical+population+f.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/collab/f2/v10/"

TARGET_COL   = "in_fire"
RANDOM_STATE = 42

# ── Grille spatiale (Portugal entier) ────────────────────────
N_BLOCKS_X, N_BLOCKS_Y = 4, 4   # 25 blocs

# ── Nested Cross-Validation spatiale ─────────────────────────
K_TEST_BLOCKS    = 3     # blocs TEST par iteration
N_CV_ITERATIONS  = 5     # iterations (chacune a son propre Optuna)

# ── Optuna (1 par iteration) ─────────────────────────────────
N_OPTUNA_TRIALS  = 10    # trials par iteration (peu car x8 iterations)
N_OPTUNA_SPLITS  = 2     # splits par trial
K_VAL_BLOCKS     = 3     # blocs val par split

# ── Divers ───────────────────────────────────────────────────
MIN_FIRES_PER_BLOCK = 0
MAX_FIRE_RATE_TRAIN = 1
RADIUS_KM = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Nested CV: {K_TEST_BLOCKS} blocs test x {N_CV_ITERATIONS} iterations")
print(f"Optuna: {N_OPTUNA_TRIALS} trials x {N_OPTUNA_SPLITS} splits PAR ITERATION")
print(f"Total Optuna fits: ~{N_CV_ITERATIONS * N_OPTUNA_TRIALS * N_OPTUNA_SPLITS}")
print(f"Grille: {N_BLOCKS_X}x{N_BLOCKS_Y} = {N_BLOCKS_X*N_BLOCKS_Y} blocs")

Nested CV: 3 blocs test x 5 iterations
Optuna: 10 trials x 2 splits PAR ITERATION
Total Optuna fits: ~100
Grille: 4x4 = 16 blocs
